# Unit execution for `success_probability.py`

This notebook is for manually checking the functionality of
`project.metric.success_probability` using **tweakable states, operators, counts, and weights**.

It covers:
- exact success probability for one operator $K$,
- raw output trace $\mathrm{Tr}(K\rho K^\dagger)$,
- success probability from binary probabilities,
- success probability from counts,
- effective retained-shot fraction for multi-branch workflows,
- batch summaries across several operators.


In [2]:
import numpy as np
import pandas as pd

import os
import sys

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from metric.success_probability import (
    success_probability_for_operator,
    output_trace_for_operator,
    success_probability_from_binary_probabilities,
    success_probability_from_counts,
    effective_success_probability,
    success_probabilities_for_operator_list,
    summarize_success_probability_results,
)

np.set_printoptions(precision=6, suppress=True)


## 1. Helper functions and tweakable manual states

In [3]:
def ket_to_density(ket: np.ndarray) -> np.ndarray:
    ket = np.asarray(ket, dtype=complex).reshape(-1)
    norm = np.linalg.norm(ket)
    if norm == 0:
        raise ValueError("ket must be nonzero.")
    ket = ket / norm
    return np.outer(ket, ket.conj())

def pretty_matrix(name, mat):
    print(f"{name} =")
    print(np.asarray(mat))
    print("trace =", np.trace(mat))
    print()

# -------- Tweak these freely --------
ket_plus = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2.0)
ket_custom = np.array([np.sqrt(0.7), np.sqrt(0.3) * np.exp(1j * np.pi / 6)], dtype=complex)

rho_pure = ket_to_density(ket_plus)
rho_custom = 0.63 * ket_to_density(ket_custom)   # subnormalised example

pretty_matrix("rho_pure", rho_pure)
pretty_matrix("rho_custom (subnormalised)", rho_custom)


rho_pure =
[[0.5+0.j 0.5+0.j]
 [0.5+0.j 0.5+0.j]]
trace = (1.0000000000000002+0j)

rho_custom (subnormalised) =
[[0.441   +0.j       0.250023-0.144351j]
 [0.250023+0.144351j 0.189   -0.j      ]]
trace = (0.63-5.79002812134572e-19j)



## 2. Tweakable manual operators

In [4]:
# A diagonal nonunitary operator
K_diag = np.diag([0.9, 0.4]).astype(complex)

# A sparse / near-diagonal operator
K_sparse = np.array([
    [0.95, 0.10],
    [0.00, 0.35],
], dtype=complex)

# Another custom operator
K_custom = np.array([
    [0.80, 0.00],
    [0.15, 0.45],
], dtype=complex)

pretty_matrix("K_diag", K_diag)
pretty_matrix("K_sparse", K_sparse)
pretty_matrix("K_custom", K_custom)


K_diag =
[[0.9+0.j 0. +0.j]
 [0. +0.j 0.4+0.j]]
trace = (1.3+0j)

K_sparse =
[[0.95+0.j 0.1 +0.j]
 [0.  +0.j 0.35+0.j]]
trace = (1.2999999999999998+0j)

K_custom =
[[0.8 +0.j 0.  +0.j]
 [0.15+0.j 0.45+0.j]]
trace = (1.25+0j)



## 3. Exact success probability and output trace for one operator

In [5]:
rho = rho_custom     # change this
K = K_diag             # change this

result = success_probability_for_operator(rho, K, label="manual_example")
trace_out = output_trace_for_operator(rho, K)

print("Success probability =", result.success_probability)
print("Input trace =", result.total_probability)
print("Output trace Tr(K rho K^dagger) =", trace_out)


Success probability = 0.6150000000000001
Input trace = 0.63
Output trace Tr(K rho K^dagger) = 0.38745000000000007


## 4. Check several operators on the same state

In [6]:
operators = [K_diag, K_sparse, K_custom]
labels = ["diag", "sparse", "custom"]

results = success_probabilities_for_operator_list(rho_custom, operators, labels=labels)

df_ops = pd.DataFrame({
    "label": [r.label for r in results],
    "success_probability": [r.success_probability for r in results],
    "input_trace": [r.total_probability for r in results],
})

df_ops


,label,success_probability,input_trace
0,diag,0.615000,0.63
1,sparse,0.746904,0.63
2,custom,0.578076,0.63


## 5. Summary across several operator results

In [7]:
summary = summarize_success_probability_results(results)
summary


SuccessProbabilitySummary(sample_size=3, mean_success_probability=0.6466601254714663, std_success_probability=0.07246815155592387, min_success_probability=0.578076464049058, max_success_probability=0.7469039123653408)

## 6. Success probability from binary probabilities

In [8]:
binary_probs = [0.62, 0.38]   # [success, failure]
res_probs = success_probability_from_binary_probabilities(
    binary_probs,
    success_index=0,
    label="binary_prob_example",
)

print("Success probability from probabilities =", res_probs.success_probability)
print("Total probability =", res_probs.total_probability)


Success probability from probabilities = 0.62
Total probability = 1.0


## 7. Success probability from counts

In [9]:
counts = {"0": 812, "1": 188}
res_counts = success_probability_from_counts(
    counts,
    success_key="0",
    label="counts_example",
)

print("Success probability from counts =", res_counts.success_probability)
print("Total shots =", res_counts.total_probability)


Success probability from counts = 0.812
Total shots = 1000.0


## 8. Effective retained-shot fraction for multi-branch workflows

In [10]:
branch_success_probabilities = [0.71, 0.46, 0.58]
branch_weights = [0.5, 0.3, 0.2]

res_eff = effective_success_probability(
    branch_success_probabilities,
    weights=branch_weights,
    label="multi_branch_example",
)

print("Effective success probability =", res_eff.success_probability)
print("Total weight =", res_eff.total_probability)


Effective success probability = 0.609
Total weight = 1.0


## 9. Minimal template for your own manual inputs

In [11]:
# Replace these with your own state and operator
rho_user = np.array([
    [0.55, 0.18 + 0.05j],
    [0.18 - 0.05j, 0.25],
], dtype=complex)

K_user = np.array([
    [0.90, 0.00],
    [0.05, 0.40],
], dtype=complex)

res_user = success_probability_for_operator(rho_user, K_user, label="user_case")
trace_user = output_trace_for_operator(rho_user, K_user)

pretty_matrix("rho_user", rho_user)
pretty_matrix("K_user", K_user)
print("User success probability =", res_user.success_probability)
print("User output trace =", trace_user)


rho_user =
[[0.55+0.j   0.18+0.05j]
 [0.18-0.05j 0.25+0.j  ]]
trace = (0.8+0j)

K_user =
[[0.9 +0.j 0.  +0.j]
 [0.05+0.j 0.4 +0.j]]
trace = (1.3+0j)

User success probability = 0.61759375
User output trace = 0.49407500000000004
